
# OLMo-2 ICL Developmental Sweep — Results Analysis

This notebook analyzes the frozen in-context-learning (ICL) developmental sweep **without loading any language-model checkpoints**.

The sweep evaluated 14 Stage-1 OLMo-2-1124-7B checkpoints on:

- 4 synthetic classification tasks
- 8 shot counts: `0, 1, 2, 3, 4, 5, 10, 20`
- 50 frozen examples per task × shot condition
- 1,600 examples per checkpoint

The main goals here are to:

1. verify that the saved sweep is complete and internally consistent;
2. characterize how ICL behavior changes with training;
3. separate output-format learning from pattern-discovery performance;
4. inspect the full shot-count dependence, not only 20-shot accuracy;
5. quantify uncertainty and paired changes where the raw JSONL files are available;
6. identify the training interval that should receive the densest RI analysis.

This notebook is analysis-only and should run on a CPU runtime.


## 1. Setup and project paths


In [ ]:

from google.colab import drive
drive.mount("/content/drive")


In [ ]:

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint

PROJECT_DIR = Path("/content/drive/MyDrive/NLP_Project/olmo_sih_dynamics")
RESULTS_DIR = PROJECT_DIR / "results" / "icl"
ANALYSIS_DIR = RESULTS_DIR / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

TRAJECTORY_PATH = RESULTS_DIR / "icl_checkpoint_trajectory_all_shots.csv"
TRAJECTORY_20_PATH = RESULTS_DIR / "icl_checkpoint_trajectory_20shot.csv"
MANIFEST_PATH = RESULTS_DIR / "selected_checkpoint_manifest.csv"

print("Results directory:", RESULTS_DIR)
print("Analysis directory:", ANALYSIS_DIR)


## 2. Load the saved sweep outputs


In [ ]:

for path in [TRAJECTORY_PATH, TRAJECTORY_20_PATH, MANIFEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing expected sweep output: {path}")

trajectory = pd.read_csv(TRAJECTORY_PATH)
trajectory_20 = pd.read_csv(TRAJECTORY_20_PATH)
manifest = pd.read_csv(MANIFEST_PATH)

manifest = manifest.sort_values(["step", "tokens_B"]).reset_index(drop=True)

checkpoint_meta = manifest[
    ["revision", "step", "tokens_B"]
].drop_duplicates()

assert checkpoint_meta["revision"].is_unique

step_map = checkpoint_meta.set_index("revision")["step"].to_dict()
token_map = checkpoint_meta.set_index("revision")["tokens_B"].to_dict()

# The old aggregate files contain checkpoint and tokens_B but not always step.
# Add step explicitly so checkpoints that share the same rounded token label
# (600/700 -> 3B and 850/900 -> 4B) remain distinct everywhere.
trajectory["step"] = trajectory["checkpoint"].map(step_map)
trajectory_20["step"] = trajectory_20["checkpoint"].map(step_map)

if trajectory["step"].isna().any() or trajectory_20["step"].isna().any():
    missing_cp = sorted(
        set(trajectory.loc[trajectory["step"].isna(), "checkpoint"])
        | set(trajectory_20.loc[trajectory_20["step"].isna(), "checkpoint"])
    )
    raise RuntimeError(
        "Some aggregate checkpoints are absent from the manifest: "
        + ", ".join(missing_cp)
    )

trajectory["step"] = trajectory["step"].astype(int)
trajectory_20["step"] = trajectory_20["step"].astype(int)

trajectory = trajectory.sort_values(
    ["step", "task", "shots"]
).reset_index(drop=True)

trajectory_20 = trajectory_20.sort_values(
    ["step", "task"]
).reset_index(drop=True)

print("Trajectory rows:", len(trajectory))
print("20-shot rows:", len(trajectory_20))
print("Checkpoints:", trajectory["checkpoint"].nunique())
print("Tasks:", trajectory["task"].nunique())
print("Shot counts:", sorted(trajectory["shots"].unique()))

display(manifest)


## 3. Integrity checks


In [ ]:

EXPECTED_TASKS = {
    "binary_fruit_month",
    "binary_furniture_profession",
    "four_class",
    "nine_class",
}
EXPECTED_SHOTS = {0, 1, 2, 3, 4, 5, 10, 20}
EXPECTED_N_PER_CONDITION = 50

EXPECTED_CHECKPOINTS = len(manifest)

assert trajectory["checkpoint"].nunique() == EXPECTED_CHECKPOINTS, (
    f"Expected {EXPECTED_CHECKPOINTS} checkpoints from the manifest, "
    f"found {trajectory['checkpoint'].nunique()} in trajectory."
)
assert set(trajectory["task"].unique()) == EXPECTED_TASKS
assert set(trajectory["shots"].unique()) == EXPECTED_SHOTS

expected_rows = (
    EXPECTED_CHECKPOINTS
    * len(EXPECTED_TASKS)
    * len(EXPECTED_SHOTS)
)
assert len(trajectory) == expected_rows, (len(trajectory), expected_rows)

if "n" in trajectory.columns:
    bad_n = trajectory.loc[
        trajectory["n"] != EXPECTED_N_PER_CONDITION
    ]
    assert bad_n.empty, f"Unexpected condition sizes:\n{bad_n}"

condition_counts = (
    trajectory.groupby(["checkpoint", "step"])
    .size()
    .rename("conditions")
    .reset_index()
)
assert (condition_counts["conditions"] == 32).all()

print(
    f"PASS: {EXPECTED_CHECKPOINTS} checkpoints × "
    f"{len(EXPECTED_TASKS)} tasks × {len(EXPECTED_SHOTS)} shot counts "
    f"= {expected_rows} aggregate conditions."
)
print(
    f"PASS: every aggregate condition contains "
    f"{EXPECTED_N_PER_CONDITION} frozen examples."
)

print("\nExact checkpoint order:")
print(
    manifest[["revision", "step", "tokens_B"]]
    .to_string(index=False)
)



## 4. Primary 20-shot developmental trajectory

The 20-shot condition is the cleanest high-context measure of pattern discovery. We first inspect it descriptively before applying any operational definition of "emergence."

Chance accuracy is task-dependent:

- binary tasks: 0.50
- four-class: 0.25
- nine-class: 1/9 ≈ 0.111


In [ ]:

CHANCE = {
    "binary_fruit_month": 0.50,
    "binary_furniture_profession": 0.50,
    "four_class": 0.25,
    "nine_class": 1 / 9,
}

table_20 = trajectory_20[
    [
        "checkpoint",
        "step",
        "tokens_B",
        "task",
        "constrained_accuracy",
        "mean_margin",
        "format_accuracy",
        "mean_label_mass",
        "mean_gold_probability",
    ]
].copy()

table_20["chance"] = table_20["task"].map(CHANCE)
table_20["accuracy_minus_chance"] = (
    table_20["constrained_accuracy"] - table_20["chance"]
)

print(
    table_20.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}",
    )
)


### Mean 20-shot performance across the four tasks


In [ ]:

mean_20 = (
    trajectory_20.groupby(
        ["checkpoint", "step", "tokens_B"],
        as_index=False,
    )
    .agg(
        mean_accuracy=("constrained_accuracy", "mean"),
        mean_margin=("mean_margin", "mean"),
        mean_format_accuracy=("format_accuracy", "mean"),
        mean_label_mass=("mean_label_mass", "mean"),
        mean_gold_probability=("mean_gold_probability", "mean"),
    )
    .sort_values("step")
    .reset_index(drop=True)
)

print(mean_20.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

mean_20.to_csv(
    ANALYSIS_DIR / "mean_20shot_across_tasks.csv",
    index=False,
)


In [ ]:

plt.figure(figsize=(11, 5))
plt.plot(mean_20["step"], mean_20["mean_accuracy"], marker="o")
plt.xscale("log")
plt.ylim(0, 1)
plt.xlabel("Stage-1 optimizer step (log scale)")
plt.ylabel("Mean 20-shot constrained accuracy")
plt.title("Mean 20-shot ICL accuracy across tasks")
plt.grid(alpha=0.25)

tick_rows = mean_20
plt.xticks(
    tick_rows["step"],
    [
        f"{int(t)}B\ns{int(s)}"
        for t, s in zip(tick_rows["tokens_B"], tick_rows["step"])
    ],
    rotation=45,
    ha="right",
)

plt.tight_layout()
plt.show()


### 20-shot accuracy by task


In [ ]:

for task in sorted(EXPECTED_TASKS):
    part = (
        trajectory_20.loc[trajectory_20["task"] == task]
        .sort_values("step")
    )

    plt.figure(figsize=(11, 5))
    plt.plot(
        part["step"],
        part["constrained_accuracy"],
        marker="o",
    )
    plt.axhline(
        CHANCE[task],
        linestyle="--",
        label=f"Chance = {CHANCE[task]:.3f}",
    )
    plt.xscale("log")
    plt.ylim(0, 1)
    plt.xlabel("Stage-1 optimizer step (log scale)")
    plt.ylabel("20-shot constrained accuracy")
    plt.title(f"20-shot accuracy — {task}")
    plt.grid(alpha=0.25)

    plt.xticks(
        part["step"],
        [
            f"{int(t)}B\ns{int(s)}"
            for t, s in zip(part["tokens_B"], part["step"])
        ],
        rotation=45,
        ha="right",
    )
    plt.legend()
    plt.tight_layout()
    plt.show()



## 5. Format learning versus pattern discovery

`format_accuracy` asks whether the model's unrestricted top token is one of the legal labels.

`mean_label_mass` asks how much total next-token probability mass the model assigns to all legal labels.

These are distinct from classification accuracy. If format metrics saturate earlier than task accuracy, that supports a developmental separation between learning the response format and learning to infer the demonstrated mapping.


In [ ]:

format_summary = (
    trajectory_20.groupby(
        ["checkpoint", "step", "tokens_B"],
        as_index=False,
    )
    .agg(
        mean_format_accuracy=("format_accuracy", "mean"),
        mean_label_mass=("mean_label_mass", "mean"),
        mean_accuracy=("constrained_accuracy", "mean"),
    )
    .sort_values("step")
    .reset_index(drop=True)
)

print(format_summary.to_string(index=False, float_format=lambda x: f"{x:.3f}"))


In [ ]:

plt.figure(figsize=(11, 5))
plt.plot(
    format_summary["step"],
    format_summary["mean_format_accuracy"],
    marker="o",
    label="Format accuracy",
)
plt.plot(
    format_summary["step"],
    format_summary["mean_label_mass"],
    marker="o",
    label="Legal-label probability mass",
)
plt.plot(
    format_summary["step"],
    format_summary["mean_accuracy"],
    marker="o",
    label="Constrained accuracy",
)
plt.xscale("log")
plt.ylim(0, 1.05)
plt.xlabel("Stage-1 optimizer step (log scale)")
plt.ylabel("Mean score across tasks")
plt.title("Format acquisition versus 20-shot task performance")
plt.grid(alpha=0.25)
plt.xticks(
    format_summary["step"],
    [
        f"{int(t)}B\ns{int(s)}"
        for t, s in zip(
            format_summary["tokens_B"],
            format_summary["step"],
        )
    ],
    rotation=45,
    ha="right",
)
plt.legend()
plt.tight_layout()
plt.show()


## 6. Full shot-count trajectories


In [ ]:

for task in sorted(EXPECTED_TASKS):
    part = (
        trajectory.loc[trajectory["task"] == task]
        .sort_values(["shots", "step"])
    )

    plt.figure(figsize=(11, 6))
    for shots in sorted(EXPECTED_SHOTS):
        shot_part = (
            part.loc[part["shots"] == shots]
            .sort_values("step")
        )
        plt.plot(
            shot_part["step"],
            shot_part["constrained_accuracy"],
            marker="o",
            label=f"{shots} shot",
        )

    plt.axhline(
        CHANCE[task],
        linestyle="--",
        label=f"Chance = {CHANCE[task]:.3f}",
    )
    plt.xscale("log")
    plt.ylim(0, 1)
    plt.xlabel("Stage-1 optimizer step (log scale)")
    plt.ylabel("Constrained accuracy")
    plt.title(f"Accuracy across shot counts — {task}")
    plt.grid(alpha=0.25)

    task_ticks = (
        part[["step", "tokens_B"]]
        .drop_duplicates()
        .sort_values("step")
    )
    plt.xticks(
        task_ticks["step"],
        [
            f"{int(t)}B\ns{int(s)}"
            for t, s in zip(
                task_ticks["tokens_B"],
                task_ticks["step"],
            )
        ],
        rotation=45,
        ha="right",
    )
    plt.legend(ncol=3)
    plt.tight_layout()
    plt.show()



## 7. Shot benefit / ICL gain

A useful descriptive measure is how much the high-shot condition improves over a low-shot condition at the same checkpoint.

We compute:

- `20-shot − 0-shot`
- `20-shot − 1-shot`
- `10-shot − 1-shot`

These are descriptive contrasts, not paired tests, because the frozen prompt IDs differ across shot-count conditions.


In [ ]:

gain_rows = []

for (checkpoint, step, tokens_B, task), group in trajectory.groupby(
    ["checkpoint", "step", "tokens_B", "task"]
):
    by_shot = group.set_index("shots")

    def value(shots, column):
        return float(by_shot.loc[shots, column])

    gain_rows.append({
        "checkpoint": checkpoint,
        "step": step,
        "tokens_B": tokens_B,
        "task": task,
        "acc_20_minus_0": (
            value(20, "constrained_accuracy")
            - value(0, "constrained_accuracy")
        ),
        "acc_20_minus_1": (
            value(20, "constrained_accuracy")
            - value(1, "constrained_accuracy")
        ),
        "acc_10_minus_1": (
            value(10, "constrained_accuracy")
            - value(1, "constrained_accuracy")
        ),
        "margin_20_minus_1": (
            value(20, "mean_margin")
            - value(1, "mean_margin")
        ),
        "label_mass_20_minus_1": (
            value(20, "mean_label_mass")
            - value(1, "mean_label_mass")
        ),
    })

shot_gains = (
    pd.DataFrame(gain_rows)
    .sort_values(["step", "task"])
    .reset_index(drop=True)
)

shot_gains.to_csv(
    ANALYSIS_DIR / "shot_gain_trajectory.csv",
    index=False,
)

print(
    shot_gains[
        [
            "checkpoint",
            "step",
            "tokens_B",
            "task",
            "acc_20_minus_0",
            "acc_20_minus_1",
            "acc_10_minus_1",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}",
    )
)


In [ ]:

for task in sorted(EXPECTED_TASKS):
    part = shot_gains.loc[shot_gains["task"] == task].sort_values("tokens_B")

    plt.figure(figsize=(9, 5))
    plt.plot(part["tokens_B"], part["acc_20_minus_1"], marker="o")
    plt.axhline(0, linestyle="--")
    plt.xscale("log")
    plt.xlabel("Stage-1 training tokens (billions, log scale)")
    plt.ylabel("Accuracy gain: 20-shot − 1-shot")
    plt.title(f"High-context ICL gain — {task}")
    plt.grid(alpha=0.25)
    plt.show()



## 8. Binomial confidence intervals for 20-shot accuracy

Each task/checkpoint 20-shot accuracy is based on 50 examples. Wilson intervals provide a useful view of sampling uncertainty.

The intervals should not be interpreted as correcting for every design choice in the experiment; they quantify binomial uncertainty for the frozen evaluation sample.


In [ ]:

ci_rows = []

for row in trajectory_20.itertuples(index=False):
    n = int(row.n)
    successes = int(round(row.constrained_accuracy * n))
    low, high = proportion_confint(
        successes,
        n,
        alpha=0.05,
        method="wilson",
    )

    ci_rows.append({
        "checkpoint": row.checkpoint,
        "step": row.step,
        "tokens_B": row.tokens_B,
        "task": row.task,
        "accuracy": row.constrained_accuracy,
        "n": n,
        "ci_low": low,
        "ci_high": high,
        "chance": CHANCE[row.task],
        "lower_above_chance": low > CHANCE[row.task],
    })

accuracy_ci_20 = (
    pd.DataFrame(ci_rows)
    .sort_values(["step", "task"])
    .reset_index(drop=True)
)

accuracy_ci_20.to_csv(
    ANALYSIS_DIR / "accuracy_20shot_wilson_ci.csv",
    index=False,
)

print(
    accuracy_ci_20.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}",
    )
)


In [ ]:

for task in sorted(EXPECTED_TASKS):
    part = accuracy_ci_20.loc[accuracy_ci_20["task"] == task].sort_values("tokens_B")
    lower_err = part["accuracy"] - part["ci_low"]
    upper_err = part["ci_high"] - part["accuracy"]

    plt.figure(figsize=(9, 5))
    plt.errorbar(
        part["tokens_B"],
        part["accuracy"],
        yerr=[lower_err, upper_err],
        marker="o",
        capsize=3,
    )
    plt.axhline(CHANCE[task], linestyle="--", label=f"Chance = {CHANCE[task]:.3f}")
    plt.xscale("log")
    plt.ylim(0, 1)
    plt.xlabel("Stage-1 training tokens (billions, log scale)")
    plt.ylabel("20-shot constrained accuracy")
    plt.title(f"20-shot accuracy with 95% Wilson CI — {task}")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()



## 9. Load raw per-example predictions for paired analyses

The aggregate CSVs are sufficient for the main trajectory. The raw checkpoint JSONLs let us compare the **same prompt ID** across training checkpoints.

This section loads only saved predictions; it does not load any OLMo model.


In [ ]:

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

raw_frames = []
missing_raw = []

for cp in manifest.itertuples(index=False):
    path = RESULTS_DIR / f"{cp.revision}.jsonl"

    if not path.exists():
        missing_raw.append(str(path))
        continue

    frame = pd.DataFrame(load_jsonl(path))
    frame["checkpoint_manifest"] = cp.revision
    frame["tokens_B_manifest"] = cp.tokens_B
    raw_frames.append(frame)

if missing_raw:
    print("WARNING: raw files missing:")
    for path in missing_raw:
        print("  ", path)

raw = pd.concat(raw_frames, ignore_index=True) if raw_frames else pd.DataFrame()

print("Raw rows loaded:", len(raw))
print("Raw checkpoints loaded:", raw["checkpoint"].nunique() if not raw.empty else 0)

if not raw.empty:
    expected_raw_rows = len(manifest) * 1600
    print("Expected if all raw files are present:", expected_raw_rows)



## 10. Consecutive-checkpoint paired transitions at 20 shots

For each adjacent pair of training checkpoints, we count:

- wrong → wrong
- wrong → correct
- correct → wrong
- correct → correct

Because the prompt IDs are identical across checkpoints, correctness changes are genuinely paired.

The exact McNemar test is implemented as a two-sided binomial test over the discordant pairs. Holm correction is reported across all task × transition tests.


In [ ]:

transition_rows = []

if raw.empty:
    print("Raw prediction files were not available; skipping paired analysis.")
else:
    raw20 = raw.loc[raw["shots"] == 20].copy()

    ordered_checkpoints = (
        manifest.sort_values("step")
        .to_dict("records")
    )

    for left, right in zip(
        ordered_checkpoints[:-1],
        ordered_checkpoints[1:],
    ):
        left_df = raw20.loc[
            raw20["checkpoint"] == left["revision"],
            ["id", "task", "constrained_correct"],
        ].rename(
            columns={"constrained_correct": "correct_left"}
        )

        right_df = raw20.loc[
            raw20["checkpoint"] == right["revision"],
            ["id", "task", "constrained_correct"],
        ].rename(
            columns={"constrained_correct": "correct_right"}
        )

        paired = left_df.merge(
            right_df,
            on=["id", "task"],
            how="inner",
        )

        for task, group in paired.groupby("task"):
            early = group["correct_left"].astype(bool)
            late = group["correct_right"].astype(bool)

            ww = int(((~early) & (~late)).sum())
            wc = int(((~early) & late).sum())
            cw = int((early & (~late)).sum())
            cc = int((early & late).sum())

            discordant = wc + cw
            p_value = (
                binomtest(
                    min(wc, cw),
                    n=discordant,
                    p=0.5,
                    alternative="two-sided",
                ).pvalue
                if discordant > 0
                else 1.0
            )

            transition_rows.append({
                "from_step": int(left["step"]),
                "to_step": int(right["step"]),
                "from_tokens_B": left["tokens_B"],
                "to_tokens_B": right["tokens_B"],
                "from_checkpoint": left["revision"],
                "to_checkpoint": right["revision"],
                "task": task,
                "wrong_to_wrong": ww,
                "wrong_to_correct": wc,
                "correct_to_wrong": cw,
                "correct_to_correct": cc,
                "net_correctness_change": wc - cw,
                "mcnemar_exact_p": p_value,
            })

    transitions = pd.DataFrame(transition_rows)

    reject, p_holm, _, _ = multipletests(
        transitions["mcnemar_exact_p"].values,
        alpha=0.05,
        method="holm",
    )
    transitions["mcnemar_holm_p"] = p_holm
    transitions["holm_significant"] = reject

    transitions.to_csv(
        ANALYSIS_DIR
        / "paired_20shot_consecutive_transitions.csv",
        index=False,
    )

    print(
        transitions[
            [
                "from_step",
                "to_step",
                "from_tokens_B",
                "to_tokens_B",
                "task",
                "wrong_to_correct",
                "correct_to_wrong",
                "net_correctness_change",
                "mcnemar_exact_p",
                "mcnemar_holm_p",
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}",
        )
    )



## 11. Transition diagnostics for choosing RI checkpoints

There is no single universally correct numerical definition of "ICL emergence." Rather than silently imposing one, this section reports several transparent landmarks:

- first sampled checkpoint whose 20-shot Wilson lower bound is above chance;
- checkpoint with maximum sampled 20-shot accuracy;
- checkpoint with maximum 20-shot accuracy-minus-chance;
- checkpoint with maximum 20-shot-versus-1-shot gain.

These are **descriptive landmarks**, not causal claims and not a replacement for visual inspection of the trajectory.


In [ ]:

landmarks = []

for task in sorted(EXPECTED_TASKS):
    ci_part = (
        accuracy_ci_20.loc[
            accuracy_ci_20["task"] == task
        ]
        .sort_values("step")
        .reset_index(drop=True)
    )

    gain_part = (
        shot_gains.loc[
            shot_gains["task"] == task
        ]
        .sort_values("step")
        .reset_index(drop=True)
    )

    above = ci_part.loc[
        ci_part["lower_above_chance"]
    ]

    if not above.empty:
        first_above_row = above.iloc[0]
        first_above_checkpoint = first_above_row["checkpoint"]
        first_above_step = int(first_above_row["step"])
        first_above_tokens_B = int(first_above_row["tokens_B"])
    else:
        first_above_checkpoint = np.nan
        first_above_step = np.nan
        first_above_tokens_B = np.nan

    peak_row = ci_part.loc[
        ci_part["accuracy"].idxmax()
    ]

    gain_row = gain_part.loc[
        gain_part["acc_20_minus_1"].idxmax()
    ]

    acc_minus_chance = (
        ci_part["accuracy"]
        - ci_part["chance"]
    )
    best_excess_row = ci_part.loc[
        acc_minus_chance.idxmax()
    ]

    landmarks.append({
        "task": task,
        "chance": CHANCE[task],

        "first_20shot_CI_above_chance_checkpoint":
            first_above_checkpoint,
        "first_20shot_CI_above_chance_step":
            first_above_step,
        "first_20shot_CI_above_chance_B":
            first_above_tokens_B,

        "peak_20shot_checkpoint":
            peak_row["checkpoint"],
        "peak_20shot_step":
            int(peak_row["step"]),
        "peak_20shot_accuracy_B":
            int(peak_row["tokens_B"]),
        "peak_20shot_accuracy":
            float(peak_row["accuracy"]),

        "max_excess_over_chance_checkpoint":
            best_excess_row["checkpoint"],
        "max_excess_over_chance_step":
            int(best_excess_row["step"]),
        "max_excess_over_chance_B":
            int(best_excess_row["tokens_B"]),
        "max_excess_over_chance":
            float(
                best_excess_row["accuracy"]
                - best_excess_row["chance"]
            ),

        "max_20minus1_gain_checkpoint":
            gain_row["checkpoint"],
        "max_20minus1_gain_step":
            int(gain_row["step"]),
        "max_20minus1_gain_B":
            int(gain_row["tokens_B"]),
        "max_20minus1_gain":
            float(gain_row["acc_20_minus_1"]),
    })

landmarks = pd.DataFrame(landmarks)

landmarks.to_csv(
    ANALYSIS_DIR / "icl_transition_landmarks.csv",
    index=False,
)

print(
    landmarks.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}",
    )
)



## 12. Compact checkpoint table for RI planning

The RI sweep should be chosen only after interpreting the plots and diagnostics above. This table puts the most relevant developmental measures side by side.

A likely useful strategy is to include:

- a clear pre-transition checkpoint;
- multiple checkpoints through the rapid-rise interval;
- one or two post-rise checkpoints;
- the final Stage-1 checkpoint as a mature reference.

Do not choose the final RI schedule from one metric alone.


In [ ]:

ri_planning = (
    trajectory_20[
        [
            "checkpoint",
            "step",
            "tokens_B",
            "task",
            "constrained_accuracy",
            "mean_margin",
            "format_accuracy",
            "mean_label_mass",
        ]
    ]
    .merge(
        shot_gains[
            [
                "checkpoint",
                "step",
                "tokens_B",
                "task",
                "acc_20_minus_1",
                "acc_20_minus_0",
            ]
        ],
        on=[
            "checkpoint",
            "step",
            "tokens_B",
            "task",
        ],
        how="left",
        validate="one_to_one",
    )
    .sort_values(["step", "task"])
    .reset_index(drop=True)
)

assert len(ri_planning) == len(trajectory_20)

ri_planning.to_csv(
    ANALYSIS_DIR / "ri_checkpoint_planning_table.csv",
    index=False,
)

print(
    ri_planning.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}",
    )
)



### Important checkpoint-identity rule

Checkpoint identity is defined by the exact **revision / optimizer step**, not by the rounded `tokens_B` label.

In particular:

- step 600 and step 700 are both labeled approximately 3B tokens;
- step 850 and step 900 are both labeled approximately 4B tokens.

Therefore no analysis, merge, ordering operation, or aggregation in this notebook groups solely on `tokens_B`. This prevents the intermediate checkpoints from being collapsed or cross-joined.



## 13. Analysis outputs

This notebook writes the following derived files to:

`NLP_Project/olmo_sih_dynamics/results/icl/analysis/`

- `mean_20shot_across_tasks.csv` — one row per exact checkpoint;
- `shot_gain_trajectory.csv` — checkpoint/step aware;
- `accuracy_20shot_wilson_ci.csv` — checkpoint/step aware;
- `paired_20shot_consecutive_transitions.csv` — ordered by exact optimizer step;
- `icl_transition_landmarks.csv` — reports exact checkpoint and step as well as token count;
- `ri_checkpoint_planning_table.csv` — one-to-one merge by exact checkpoint identity.

The dense early checkpoints (600, 700, 850, 900, 1000) remain distinct even when multiple revisions share the same rounded billion-token label.
